In [29]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [30]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(
        self,
        rope_dim: int,
        max_seq_len: int,
        base: float = 10000.0,
    ):
        super().__init__()

        assert rope_dim % 2 == 0, "rope_dim must be even for RoPE"

        self.rope_dim = rope_dim
        self.max_seq_len = max_seq_len

        inv_freq = torch.exp(
            torch.arange(0, rope_dim, 2, dtype=torch.float32)
            * (-math.log(base) / rope_dim)
        )
        # [rope_dim / 2]

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        ).unsqueeze(1)
        # [max_seq_len, 1]

        angles = positions * inv_freq
        # [max_seq_len, rope_dim / 2]

        cos = torch.cos(angles)
        sin = torch.sin(angles)

        # Broadcast with x: [B, H, T, rope_dim]
        self.register_buffer("cos", cos[None, None, :, :])
        self.register_buffer("sin", sin[None, None, :, :])

    def forward(self, x):
        """
        x: [B, H, T, rope_dim]

        returns:
            x_rotated: [B, H, T, rope_dim]
        """

        batch_size, n_heads, seq_len, rope_dim = x.shape

        assert rope_dim == self.rope_dim
        assert seq_len <= self.max_seq_len

        cos = self.cos[:, :, :seq_len, :]
        sin = self.sin[:, :, :seq_len, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd = x_even * sin + x_odd * cos

        x_rot = torch.empty_like(x)
        x_rot[..., 0::2] = x_rot_even
        x_rot[..., 1::2] = x_rot_odd

        return x_rot

In [31]:
def apply_rope_bthd(
    rope: RotaryPositionalEmbedding,
    x: torch.Tensor,
    positions: torch.Tensor,
):
    """
    Applies your consistent RoPE implementation to tensors shaped [B, T, H, D].

    rope.forward expects:
        [B, H, T, D]

    This helper accepts:
        [B, T, H, D]

    positions:
        [T] absolute positions
    """

    # [B, T, H, D] -> [B, H, T, D]
    x = x.transpose(1, 2)

    # Case 1: positions are exactly [0, 1, 2, ..., T-1]
    # Then we can directly use your rope.forward().
    if torch.equal(
        positions,
        torch.arange(x.shape[2], device=positions.device),
    ):
        x = rope(x)

    else:
        # Cached inference case:
        # positions may be [prompt_len], [prompt_len + 1], etc.
        B, H, T, D = x.shape

        cos = rope.cos[:, :, positions, :]
        sin = rope.sin[:, :, positions, :]

        # cos/sin shape becomes [1, 1, T, D/2]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd = x_even * sin + x_odd * cos

        x_rot = torch.empty_like(x)
        x_rot[..., 0::2] = x_rot_even
        x_rot[..., 1::2] = x_rot_odd

        x = x_rot

    # [B, H, T, D] -> [B, T, H, D]
    x = x.transpose(1, 2)

    return x

In [32]:
# ============================================================
# Block 2: MLA module (paper-faithful core)
# ------------------------------------------------------------
# We follow the common MLA equations:
#
#   c_t^KV = W_DKV x_t                (KV low-rank latent)
#   k_t,i^C = W_UK_i c_t^KV           (per-head key content part)
#   v_t,i^C = W_UV_i c_t^KV           (per-head value content part)
#
#   c_t^Q  = W_DQ x_t                 (Q low-rank latent, for training memory)
#   q_t,i^C = W_UQ_i c_t^Q            (per-head query content part)
#
# Decoupled RoPE:
#   k_t^R = RoPE(W_KR x_t, t)         (shared RoPE key, same for all heads)
#   q_t,i^R = RoPE(W_QR_i c_t^Q, t)   (per-head RoPE query)
#
# Concatenate for attention logits:
#   k_t,i = [k_t,i^C ; k_t^R],  q_t,i = [q_t,i^C ; q_t,i^R]
# Attention weights use dimension (d_nope + d_rope), but values are only v^C (d_nope).
#
# Cache during decoding: ONLY {c_t^KV, k_t^R} per token (small).
# ============================================================

class MLAAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_nope: int,          # content (non-RoPE) per-head dim
        d_rope: int,          # RoPE per-head dim (even)
        r_kv: int,            # KV latent rank
        r_q: int,             # Q latent rank
    ):
        super().__init__()
        assert d_rope % 2 == 0, "d_rope must be even for RoPE."
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_nope = d_nope
        self.d_rope = d_rope
        self.d_attn = d_nope + d_rope
        self.r_kv = r_kv
        self.r_q = r_q

        # KV low-rank compression/decompression
        # what it does: compress the token hidden state into a small KV latent.  D stands for down projection
        self.W_DKV = nn.Linear(d_model, r_kv, bias=False)

        # what it does: “inflate” latent into content keys (no positional part) for all heads: U stands for up projection
        self.W_UK  = nn.Linear(r_kv, num_heads * d_nope, bias=False)

        # What it does: inflate latent into content values:
        self.W_UV  = nn.Linear(r_kv, num_heads * d_nope, bias=False)

        # Q low-rank compression/decompression
        # What it does: compress token hidden state into a query latent
        self.W_DQ  = nn.Linear(d_model, r_q, bias=False)

        # What it does: inflate query latent in to content queries 
        self.W_UQ  = nn.Linear(r_q, num_heads * d_nope, bias=False)

        # Decoupled RoPE projections
        # What it does: produces a small RoPE key vector
        self.W_KR = nn.Linear(d_model, d_rope, bias=False)                 # shared across heads

        # what it does: produces rope query components
        self.W_QR = nn.Linear(r_q, num_heads * d_rope, bias=False)         # per head, from c^Q

        # Output projection (note: we output H * d_nope, not including RoPE dims)

        # What it does: After attention, each head outputs a vector in 𝑅𝑑nope. Concatenate heads to 𝑅𝐻𝑑nope, then map back to model dim
        self.W_O = nn.Linear(num_heads * d_nope, d_model, bias=False)

        self.rope = RotaryPositionalEmbedding(
            rope_dim=d_rope,
            max_seq_len=max_seq_len,
        )


    def forward(
        self,
        x: torch.Tensor,                     # [B, T, D]
        positions: torch.Tensor,             # [T] absolute positions for these tokens
        attn_mask: torch.Tensor = None,      # [1,1,T,S] additive mask (e.g., causal for full seq)
        cache: tuple = None,                 # (c_kv_cache [B,S_old,r_kv], k_rope_cache [B,S_old,1,d_rope])
    ):
        B, T, D = x.shape
        assert D == self.d_model

        # -------------------------
        # 1) Build Q parts
        # -------------------------
        # Refer to Imp Commands for undersatnding the transformation
        # create query latent
        # c^Q = W_DQ x 
        c_q = self.W_DQ(x)                                            # [B,T,r_q]

        # from the small query latent, generate per-head content query vectors.
        # q^C = W_UQ c^Q  -> reshape to heads
        q_c = self.W_UQ(c_q).view(B, T, self.num_heads, self.d_nope)   # [B,T,H,d_nope]

        # build the per-head “positional” part of the queries and rotate it based on token positions
        # q^R = RoPE(W_QR c^Q)
        q_r = self.W_QR(c_q).view(B, T, self.num_heads, self.d_rope)   # [B,T,H,d_rope]
        q_r = apply_rope_bthd(
                rope=self.rope,
                x=q_r,
                positions=positions,
                )


        # the final per-head query vector is: first d_nope dims = content , last d_rope dims = positional
        # q = [q^C ; q^R]
        q = torch.cat([q_c, q_r], dim=-1)                              # [B,T,H,d_attn]

        # -------------------------
        # 2) Build KV parts for *new* tokens
        # -------------------------

        # create kv latent
        # c^KV = W_DKV x
        c_kv_new = self.W_DKV(x)                                       # [B,T,r_kv]

        # from the small kv latent, generate per-head content key and value vectors.
        # k^C, v^C from c^KV
        k_c_new = self.W_UK(c_kv_new).view(B, T, self.num_heads, self.d_nope)  # [B,T,H,d_nope]
        v_c_new = self.W_UV(c_kv_new).view(B, T, self.num_heads, self.d_nope)  # [B,T,H,d_nope]

        # shared RoPE key across all heads : k^R = RoPE(W_KR x)  Note: Use original token embeddings not latent state as we did for query
        k_r_new = self.W_KR(x).view(B, T, 1, self.d_rope)              # [B,T,1,d_rope]
        k_r_new = apply_rope_bthd(
            rope=self.rope,
            x=k_r_new,
            positions=positions,
        )

        # Expand shared key across heads for attention math 
        # .expand(-1, -1, self.num_heads, -1) -1 means keep the first, second and lst dimensions same, bradcast, 3 dimension 
        k_r_new_exp = k_r_new.expand(-1, -1, self.num_heads, -1)        # [B,T,H,d_rope]
        k_new = torch.cat([k_c_new, k_r_new_exp], dim=-1)               # [B,T,H,d_attn]

        # -------------------------
        # 3) If cache exists, prepend cached prefix K/V (naive reconstruction)
        # -------------------------
        if cache is not None:
            c_kv_cache, k_r_cache = cache                               # c_kv_cache:[B,S0,r_kv], k_r_cache:[B,S0,1,d_rope]
            S0 = c_kv_cache.shape[1]

            # Reconstruct cached content keys/values from cached latent (naive form).
            # (DeepSeek discusses an "absorb" optimization that can avoid this reconstruction cost.)
            # When you generate tokens one-by-one, you don’t want to recompute keys/values for the whole prefix every step. So you cache some stuff from previous steps. In MLA, what you cache is:
            # c_kv_cache: the KV latent for past tokens (small)
            # k_r_cache: the shared RoPE key for past tokens (small)
            # Then, for the current step, you compute c_kv_new, k_new, v_c_new for the new token(s) and prepend the cached prefix so attention can see the whole prefix.
            
            k_c_cache = self.W_UK(c_kv_cache).view(B, S0, self.num_heads, self.d_nope)
            v_c_cache = self.W_UV(c_kv_cache).view(B, S0, self.num_heads, self.d_nope)

            k_r_cache_exp = k_r_cache.expand(-1, -1, self.num_heads, -1)
            k_cache = torch.cat([k_c_cache, k_r_cache_exp], dim=-1)      # [B,S0,H,d_attn]

            # Values are only content part (d_nope)
            v_c = torch.cat([v_c_cache, v_c_new], dim=1)                 # [B,S0+T,H,d_nope]
            k_all = torch.cat([k_cache, k_new], dim=1)                   # [B,S0+T,H,d_attn]
        else:
            v_c = v_c_new                                                # [B,T,H,d_nope]
            k_all = k_new                                                # [B,T,H,d_attn]

        # -------------------------
        # 4) Attention: softmax(q k^T / sqrt(d_nope+d_rope)) * v^C
        # -------------------------
        # scores: [B,H,T,S]
        
        # torch.einsum -> firts maps the dimensions of tensors to bthd,bshd and repesctively and does 
         # permutation and matrix multiplication
        
        # 1. Permute both tensors to bring batch and heads to the front
        # q_perm = q.permute(0, 2, 1, 3)          # Shape: [b, h, t, d]
        # k_perm = k_all.permute(0, 2, 3, 1)      # Shape: [b, h, d, s] (transposed d and s for matmul)
        
        # 2. Perform batch matrix multiplication and scale
        # scores = torch.matmul(q_perm, k_perm) / math.sqrt(self.d_attn) # Shape: [b, h, t, s]
        
        scores = torch.einsum("bthd,bshd->bhts", q, k_all) / math.sqrt(self.d_attn)
        if attn_mask is not None:
            scores = scores + attn_mask 
        
        # Notes this is additive attention mask which is different from casual boolean mask 
        # For MLA with cache, additive masks are often easier because query length T and key length S may be different
        
        attn = torch.softmax(scores, dim=-1)                             # [B,H,T,S]
        out = torch.einsum("bhts,bshd->bthd", attn, v_c)                  # [B,T,H,d_nope]

        # -------------------------
        # 5) Output projection
        # -------------------------
        out = out.reshape(B, T, self.num_heads * self.d_nope)            # [B,T,H*d_nope]
        y = self.W_O(out)                                                # [B,T,D]

        # Cache only NEW pieces for these tokens (append outside)
        new_cache_piece = (c_kv_new.detach(), k_r_new.detach())          # [B,T,r_kv], [B,T,1,d_rope]
        return y, new_cache_piece


In [33]:
# ============================================================
# Block 3: Utility to build a causal mask for "full-seq" mode
# ------------------------------------------------------------
# When we feed a whole sequence at once, we must prevent token t from attending
# to future tokens > t. We do that by adding a large negative mask above diagonal.
# ============================================================

def causal_mask(Tq: int, Tk: int, device=device):
    # returns [1,1,Tq,Tk]
    m = torch.triu(torch.ones(Tq, Tk, device=device) * (-1e9), diagonal=1)
    return m[None, None, :, :]


In [ ]:
# Helper 

def build_additive_causal_mask(
    positions,
    key_len,
    dtype,
    device,
    key_padding_mask=None,
):
    T = positions.shape[0]

    key_positions = torch.arange(key_len, device=device)

    allowed = key_positions[None, :] <= positions[:, None]
    # [T, S]

    additive_mask = torch.zeros(
        1,
        1,
        T,
        key_len,
        device=device,
        dtype=dtype,
    )

    additive_mask = additive_mask.masked_fill(
        ~allowed[None, None, :, :],
        torch.finfo(dtype).min,
    )

    if key_padding_mask is not None:
        B = key_padding_mask.shape[0]

        additive_mask = additive_mask.expand(
            B,
            -1,
            -1,
            -1,
        ).clone()

        additive_mask = additive_mask.masked_fill(
            key_padding_mask[:, None, None, :].bool() == 0,
            torch.finfo(dtype).min,
        )

    return additive_mask

In [34]:
# ============================================================
# Block 4: End-to-end demo (full sequence vs incremental decode)
# ------------------------------------------------------------
# We verify correctness in the simplest way:
#  1) Run MLA on the full sequence with a causal mask.
#  2) Run MLA token-by-token, caching only (c_kv, k_rope).
# If MLA is implemented correctly, both outputs match exactly (up to fp error).
# ============================================================

# Tiny toy dims so it's fast and easy to inspect
B, T = 2, 6
d_model = 32
num_heads = 4
d_nope = 4
d_rope = 4
r_kv = 8
r_q  = 8

#In MLA, query and KV compression dimensions are separate because they optimize different bottlenecks. 
#The KV latent dimension directly controls how much information must be cached per token during generation, 
# so it is central to memory savings. The query latent dimension only affects how the current query is computed, 
# since queries are not cached in the same way. DeepSeek therefore uses separate hyperparameters for Q and KV compression 
# rather than forcing them to be equal.

mla = MLAAttention(d_model, num_heads, d_nope, d_rope, r_kv, r_q).to(device).eval()
x = torch.randn(B, T, d_model, device=device)

# ---- (A) Full sequence mode ----
pos_full = torch.arange(T, device=device)
mask_full = causal_mask(T, T, device=device)
y_full, _ = mla(x, positions=pos_full, attn_mask=mask_full, cache=None)

# ---- (B) Incremental decoding mode ----
c_cache = None
krope_cache = None
ys = []
for t in range(T):
    xt = x[:, t:t+1, :]                         # [B,1,D]
    pos_t = torch.tensor([t], device=device)    # absolute position for this token

    cache_in = None if c_cache is None else (c_cache, krope_cache)
    y_t, (c_new, krope_new) = mla(xt, positions=pos_t, attn_mask=None, cache=cache_in)

    ys.append(y_t)

    # append cache pieces
    if c_cache is None:
        c_cache = c_new
        krope_cache = krope_new
    else:
        c_cache = torch.cat([c_cache, c_new], dim=1)
        krope_cache = torch.cat([krope_cache, krope_new], dim=1)

y_inc = torch.cat(ys, dim=1)

max_err = (y_full - y_inc).abs().max().item()
print("max |y_full - y_inc| =", max_err)


max |y_full - y_inc| = 7.450580596923828e-08


In [35]:
# ============================================================
# Block 5: KV-cache size comparison (elements per token, per layer)
# ------------------------------------------------------------
# This is the main motivation for MLA:
#  - Standard MHA caches K and V: 2 * H * d_head elements per token.
#  - MLA caches only c^KV (rank r_kv) plus a shared RoPE key k^R (d_rope):
#        r_kv + d_rope elements per token (per layer).
# ============================================================

def cache_elems_per_token_mha(num_heads: int, d_head: int) -> int:
    return 2 * num_heads * d_head

def cache_elems_per_token_mla(r_kv: int, d_rope: int) -> int:
    return r_kv + d_rope

d_head_mha = d_nope + d_rope  # if you used RoPE on full head in MHA
print("MHA cache elems/token/layer:", cache_elems_per_token_mha(num_heads, d_head_mha))
print("MLA cache elems/token/layer:", cache_elems_per_token_mla(r_kv, d_rope))
print("reduction factor ~", cache_elems_per_token_mha(num_heads, d_head_mha) / cache_elems_per_token_mla(r_kv, d_rope))


MHA cache elems/token/layer: 64
MLA cache elems/token/layer: 12
reduction factor ~ 5.333333333333333
